# Fig. 2 and Sup. Figs 2-6: GSDensity base scores of the mouse brain section

Per-spot GSDensity relevance (MCA co-embedding, 300-NN graph, random walk with restart; nn_use 300, restart 0.75, MCA
dimensions 1-10, seed 0; Liang et al. 2023) of every signature of the brain figures, run once for the section in the R
environment gsdensity_env: the region top-10 signatures, the gene-set-size series of Sup. Figs 5-6, the PanglaoDB sets
of Sup. Fig. 3c and the housekeeping sets of Sup. Fig. 3a. One CSV per signature is written to data/gsd_cache/brain/.

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import json
import time

from scipy.sparse import issparse
from sklearn.metrics import roc_auc_score, average_precision_score

from helpers import (
    present_in_data,
    describe_brain_gsd_signature,
    export_gsdensity_task,
    run_gsdensity,
    read_gsdensity,
    gsdensity_signature_table,
)

In [3]:
sc.set_figure_params(frameon=False, fontsize=8)
sc.settings._vector_friendly = False
spot_size = 20

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
os.makedirs(project_dir + "logs", exist_ok=True)
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)
sc.settings.figdir = figure_dir

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"


# GSDensity scores of this dataset (one CSV per signature) and the R task directory
gsd_dir = working_dir + "data/gsd_cache/brain/"
task_dir = gsd_dir + "_tasks/section/"

In [5]:
# 1. Load data and signatures (01_visium_platform cells 6-11)
adata = sc.read_h5ad(processed_data + "visium_brain_adata.h5ad")
var = pd.Index(adata.var_names)

cell_type_1 = "Pyramidal_layer"
cell_type_2 = "Striatum"
regions = [cell_type_1, cell_type_2]

with open(f"{cell_type_1}_signatures.pkl", "rb") as f:
    signature_dict_pyramidal = pickle.load(f)
with open(f"{cell_type_2}_signatures.pkl", "rb") as f:
    signature_dict_striatum = pickle.load(f)
signature_pickles = {cell_type_1: signature_dict_pyramidal, cell_type_2: signature_dict_striatum}

print(f"{adata.n_obs} spots x {adata.n_vars} genes")
print(adata.obs["cluster"].value_counts().loc[regions].to_string())

2688 spots x 18078 genes
cluster
Pyramidal_layer     42
Striatum           153


In [6]:
# 2. Gene sets as given in the original notebooks (before matching to the data)
# gene set sizes of Fig. 2 / Sup. Figs 2, 5, 6 (01_visium_platform cells 11, 13, 43-50): every key of the pickles
size_sets = {k: list(v) for ct in regions for k, v in signature_pickles[ct].items()}

# Gene sets of Extended Data Fig. 3c (ExtendedDataFig3c_signatures); Excitatory neurons = positive control
panglao = json.load(open(working_dir + "data/F1/ED3c_gene_sets.json"))

# housekeeping genes of Sup. Fig. 3a (01_visium_platform cell 57, as written there)
housekeeping_genes = ["Gapdh", "Atp5f1", "Ubc", "Pgk1", "Tbp", "Rer1", "Hprt1", "Rpl27", "Rlp13a", "Sdha", "Actb", "B2m"]

# gene set names run through GSDensity (R column names: letters, digits and underscores only)
given_sets = {
    **size_sets,
    **{ct.replace(" ", "_"): list(v) for ct, v in panglao.items()},
    "Housekeeping": housekeeping_genes,
    f"{cell_type_1}_and_Housekeeping": size_sets[f"{cell_type_1}_top10"] + housekeeping_genes,
    f"{cell_type_2}_and_Housekeeping": size_sets[f"{cell_type_2}_top10"] + housekeeping_genes,
}

gsd_sets = {k: present_in_data(v, var) for k, v in given_sets.items()}
hk_missing = [g for g in housekeeping_genes if g not in var]
print(f"housekeeping genes absent from var_names: {hk_missing}")
print(pd.DataFrame({
    "given": {k: len(v) for k, v in given_sets.items()},
    "unique given": {k: len(set(v)) for k, v in given_sets.items()},
    "in data": {k: len(v) for k, v in gsd_sets.items()},
}).to_string())

housekeeping genes absent from var_names: ['Hprt1', 'Rlp13a']
                                  given  unique given  in data
Pyramidal_layer_top2                  2             2        2
Pyramidal_layer_top5                  5             5        5
Pyramidal_layer_top10                10            10       10
Pyramidal_layer_top20                20            20       20
Pyramidal_layer_top50                50            50       50
Pyramidal_layer_top100              100           100      100
Pyramidal_layer_top200              200           200      200
Pyramidal_layer_top500              500           500      500
Striatum_top2                         2             2        2
Striatum_top5                         5             5        5
Striatum_top10                       10            10       10
Striatum_top20                       20            20       20
Striatum_top50                       50            50       50
Striatum_top100                     100           100   

In [7]:
# 3. Signature names of the figure notebooks -> GSDensity gene set (signature = file name, gene set run, name in the
# notebook, used in)
signature_map = [
    (cell_type_1, f"{cell_type_1}_top10", cell_type_1, "Fig. 2; Sup. Figs 2, 3a, 4"),
    (cell_type_2, f"{cell_type_2}_top10", cell_type_2, "Fig. 2; Sup. Figs 2, 3b, 4"),
    *[(k, k, k, "Sup. Figs 5, 6") for k in size_sets],
    *[(ct.replace(" ", "_"), ct.replace(" ", "_"), ct, "Sup. Fig. 3c") for ct in panglao],
    ("Housekeeping", "Housekeeping", "Housekeeping", "Sup. Fig. 3a"),
    (f"{cell_type_1}+Housekeeping", f"{cell_type_1}_and_Housekeeping", f"{cell_type_1}+Housekeeping", "Sup. Fig. 3a"),
    (f"{cell_type_2}+Housekeeping", f"{cell_type_2}_and_Housekeeping", f"{cell_type_2}+Housekeeping", "Sup. Fig. 3b"),
]
signature_map = pd.DataFrame(signature_map, columns=["signature", "gene_set", "name_in_notebook", "used_in"])
signature_map["note"] = signature_map.apply(
    describe_brain_gsd_signature, axis=1, args=(size_sets, panglao, housekeeping_genes, hk_missing)
)
print(signature_map[["signature", "gene_set", "used_in"]].to_string(index=False))

                   signature                         gene_set                    used_in
             Pyramidal_layer            Pyramidal_layer_top10 Fig. 2; Sup. Figs 2, 3a, 4
                    Striatum                   Striatum_top10 Fig. 2; Sup. Figs 2, 3b, 4
        Pyramidal_layer_top2             Pyramidal_layer_top2             Sup. Figs 5, 6
        Pyramidal_layer_top5             Pyramidal_layer_top5             Sup. Figs 5, 6
       Pyramidal_layer_top10            Pyramidal_layer_top10             Sup. Figs 5, 6
       Pyramidal_layer_top20            Pyramidal_layer_top20             Sup. Figs 5, 6
       Pyramidal_layer_top50            Pyramidal_layer_top50             Sup. Figs 5, 6
      Pyramidal_layer_top100           Pyramidal_layer_top100             Sup. Figs 5, 6
      Pyramidal_layer_top200           Pyramidal_layer_top200             Sup. Figs 5, 6
      Pyramidal_layer_top500           Pyramidal_layer_top500             Sup. Figs 5, 6
               Striat

In [8]:
# 4. Export one task for the section (all gene sets share one MCA embedding and graph)
gsd_params = {"nn_use": 300, "restart": 0.75, "dims": 10, "seed": 0, "spatial": True, "spatial_n": 10,
              "spatial_ntimes": 100}
t0 = time.time()
export_gsdensity_task(adata, gsd_sets, task_dir, gsd_params)
print(f"exported {adata.n_obs} spots x {adata.n_vars} genes, {len(gsd_sets)} gene sets in {time.time() - t0:.1f} s")

exported 2688 spots x 18078 genes, 23 gene sets in 6.4 s


In [9]:
# 5. Run GSDensity (R environment gsdensity_env, one R session)
gsd_rscript = working_dir + "scripts/run_gsdensity.R"
log = working_dir + "logs/REV_GSD_base_brain_R.log"
t0 = time.time()
rc = run_gsdensity([task_dir], gsd_rscript, n_jobs=1, log=log)
wall_s = time.time() - t0
print(f"R exit codes: {rc}; wall clock: {wall_s:.1f} s")
print("".join(l for l in open(log.replace(".log", "_0.log")) if l.startswith("[gsdensity]")))

R exit codes: [0]; wall clock: 139.3 s
[gsdensity] /home/cenk/Celik_enrichmap/revision/data/gsd_cache/brain/_tasks/section/ done in 111.9s (mca 24.5s, rwr 19.4s)



In [10]:
# 6. Read the results: sum of relevance, zero relevance, antimode-positive spots and NA labels per gene set
gsd = read_gsdensity(task_dir, adata.obs_names)
scores, labels, timing = gsd["scores"], gsd["labels"], gsd["timing"]

check_rows = []
for k in gsd_sets:
    check_rows.append({
        "gene_set": k,
        "sum relevance": scores[k].sum(),
        "n zero": int((scores[k] == 0).sum()),
        "n positive (antimode)": int((labels[k] == "positive").sum()),
        "label NA": int(labels[k].isna().sum()),
    })
checks = pd.DataFrame(check_rows).set_index("gene_set")
print(checks.round(6).to_string())
print(f"timing: {timing}")

                                  sum relevance  n zero  n positive (antimode)  label NA
gene_set                                                                                
Pyramidal_layer_top2                        1.0    2047                    185         0
Pyramidal_layer_top5                        1.0    2058                    144         0
Pyramidal_layer_top10                       1.0    2049                    193         0
Pyramidal_layer_top20                       1.0    2015                    160         0
Pyramidal_layer_top50                       1.0    1895                    181         0
Pyramidal_layer_top100                      1.0    1801                    158         0
Pyramidal_layer_top200                      1.0    1566                    158         0
Pyramidal_layer_top500                      1.0    1576                    173         0
Striatum_top2                               1.0    1878                    155         0
Striatum_top5        

In [11]:
# 7. Genes used (MCA keeps genes with non-zero variance; RWR seeds = set genes in the co-embedding graph)
X = adata.X.tocsc() if issparse(adata.X) else adata.X
mean = np.asarray(X.mean(axis=0)).ravel()
sq_mean = np.asarray(X.multiply(X).mean(axis=0)).ravel() if issparse(X) else (X**2).mean(axis=0)
varying = set(var[(sq_mean - mean**2) > 0])
genes_used = {k: [g for g in v if g in varying] for k, v in gsd_sets.items()}
print(f"genes with zero variance: {adata.n_vars - len(varying)}")

genes with zero variance: 0


Two native properties of GSDensity are kept as it produces them and counted in the index: `dnet::dRWR` sets
steady-state probabilities below 1e-6 to zero, so many spots have relevance exactly 0 (ties), and `compute.cell.label`
returns NA for every spot when `multimode::locmodes` finds one mode only (no antimode).

In [12]:
# 8. One CSV per signature (spot_id, relevance, antimode label, relevance x spots) and the dataset index
runtime_note = (
    f"total {timing['total']:.1f} s (load {timing['load']:.1f}, MCA {timing['mca']:.1f}, kNN {timing['nn']:.1f}, "
    f"RWR {timing['rwr']:.1f}, spatial KLD {timing['spatial_kld']:.1f}); one run shared by all "
    f"{len(gsd_sets)} gene sets of this notebook (runtime is not per signature)"
)
subsample_note = f"none: all {adata.n_obs} spots, one MCA embedding for the section; relevance sums to 1"

index_rows = []
for _, r in signature_map.iterrows():
    tab = gsdensity_signature_table(scores, labels, r.gene_set, adata.obs_names)
    tab.to_csv(gsd_dir + f"{r.signature}.csv", index=False)
    n_used = len(genes_used[r.gene_set])
    n_given = len(set(given_sets[r.gene_set]))
    index_rows.append({
        "dataset": "brain",
        "signature": r.signature,
        "name_in_notebook": r.name_in_notebook,
        "used_in": r.used_in,
        "gene_set_run": r.gene_set,
        "source": "REV_GSD_base_brain (data/gsd_cache/brain/_tasks/section)",
        "genes_used": n_used,
        "genes_used_of_given": f"{n_used}/{n_given}",
        "genes_given": n_given,
        "genes_in_data": len(gsd_sets[r.gene_set]),
        "n_spots": len(tab),
        "n_positive_label": int((tab["label"] == "positive").sum()),
        "n_label_NA": int(tab["label"].isna().sum()),
        "label_note": (
            "label NA for all spots: compute.cell.label found one mode only (multimode::locmodes gives no "
            "antimode); native GSDensity behaviour" if tab["label"].isna().all() else ""
        ),
        "n_zero_relevance": int((tab["relevance"] == 0).sum()),
        "zero_relevance_note": "dnet::dRWR sets probabilities < 1e-6 to 0 (ties at 0)",
        "subsample_note": subsample_note,
        "runtime_s": round(timing["total"], 1),
        "runtime_note": runtime_note,
        "params": json.dumps(gsd_params),
        "note": r.note,
        "file": f"data/gsd_cache/brain/{r.signature}.csv",
    })
index = pd.DataFrame(index_rows)
index.to_csv(working_dir + "data/gsd_cache/INDEX_brain.csv", index=False)
print(index[["signature", "gene_set_run", "genes_used_of_given", "n_spots", "n_positive_label", "runtime_s", "file"]]
      .to_string(index=False))

                   signature                     gene_set_run genes_used_of_given  n_spots  n_positive_label  runtime_s                                                  file
             Pyramidal_layer            Pyramidal_layer_top10               10/10     2688               193      111.9              data/gsd_cache/brain/Pyramidal_layer.csv
                    Striatum                   Striatum_top10               10/10     2688               160      111.9                     data/gsd_cache/brain/Striatum.csv
        Pyramidal_layer_top2             Pyramidal_layer_top2                 2/2     2688               185      111.9         data/gsd_cache/brain/Pyramidal_layer_top2.csv
        Pyramidal_layer_top5             Pyramidal_layer_top5                 5/5     2688               144      111.9         data/gsd_cache/brain/Pyramidal_layer_top5.csv
       Pyramidal_layer_top10            Pyramidal_layer_top10               10/10     2688               193      111.9        dat

In [13]:
# 9. Pathway-level spatial KLD (compute.spatial.kld, 10 x 10 grid, 100 spot shuffles)
kld = gsd["spatial_kld"].loc[list(gsd_sets)].copy()
kld.index.name = "gene_set"
print(kld.round(4).to_string())

                                  spatial.kld  rspatial.kld  rspatial.kld.sd  p.value  delta.kld  p.adj
gene_set                                                                                               
Pyramidal_layer_top2                 -13.1691      -16.4524           0.1914      0.0     3.2833    0.0
Pyramidal_layer_top5                 -13.1730      -16.3673           0.2079      0.0     3.1943    0.0
Pyramidal_layer_top10                -13.1766      -16.3925           0.1848      0.0     3.2159    0.0
Pyramidal_layer_top20                -13.1917      -16.3784           0.2107      0.0     3.1867    0.0
Pyramidal_layer_top50                -13.2377      -16.4516           0.1906      0.0     3.2139    0.0
Pyramidal_layer_top100               -13.3293      -16.4857           0.1983      0.0     3.1563    0.0
Pyramidal_layer_top200               -13.4872      -16.6199           0.1977      0.0     3.1327    0.0
Pyramidal_layer_top500               -13.6851      -16.8377     

In [14]:
# 10. Sanity check: GSDensity relevance as a classifier of the annotated region (AUROC, AUPRC)
summary_rows = []
for _, r in index.iterrows():
    if r.gene_set_run in size_sets or r.gene_set_run.endswith("_and_Housekeeping"):
        ct = r.gene_set_run.split("_top")[0].split("_and_")[0]
    elif r.gene_set_run == "Housekeeping":
        ct = cell_type_1
    else:
        continue
    y = (adata.obs["cluster"] == ct).values.astype(int)
    relevance = scores[r.gene_set_run].values
    positive = (labels[r.gene_set_run] == "positive").values
    summary_rows.append({
        "signature": r.signature,
        "region": ct,
        "genes used": r.genes_used,
        "AUROC": roc_auc_score(y, relevance),
        "AUPRC": average_precision_score(y, relevance),
        "prevalence": y.mean(),
        "positive labels in region": int((positive & (y == 1)).sum()),
        "positive labels": int(positive.sum()),
    })
region_check = pd.DataFrame(summary_rows)
print(region_check.round(3).to_string(index=False))

                   signature          region  genes used  AUROC  AUPRC  prevalence  positive labels in region  positive labels
             Pyramidal_layer Pyramidal_layer          10  0.997  0.863       0.016                         42              193
                    Striatum        Striatum          10  0.977  0.828       0.057                        120              160
        Pyramidal_layer_top2 Pyramidal_layer           2  0.998  0.899       0.016                         42              185
        Pyramidal_layer_top5 Pyramidal_layer           5  0.997  0.830       0.016                         42              144
       Pyramidal_layer_top10 Pyramidal_layer          10  0.997  0.863       0.016                         42              193
       Pyramidal_layer_top20 Pyramidal_layer          20  0.997  0.851       0.016                         42              160
       Pyramidal_layer_top50 Pyramidal_layer          50  0.986  0.337       0.016                         42  

In [15]:
# 11. Summary
print(f"{len(index)} signature files from {len(gsd_sets)} GSDensity gene sets, {adata.n_obs} spots, no subsampling")
print(f"GSDensity: {runtime_note}; wall clock {wall_s:.1f} s")
print(region_check.set_index("signature")[["AUROC", "AUPRC"]].round(3).to_string())

25 signature files from 23 GSDensity gene sets, 2688 spots, no subsampling
GSDensity: total 111.9 s (load 22.2, MCA 24.5, kNN 14.9, RWR 19.4, spatial KLD 18.4); one run shared by all 23 gene sets of this notebook (runtime is not per signature); wall clock 139.3 s
                              AUROC  AUPRC
signature                                 
Pyramidal_layer               0.997  0.863
Striatum                      0.977  0.828
Pyramidal_layer_top2          0.998  0.899
Pyramidal_layer_top5          0.997  0.830
Pyramidal_layer_top10         0.997  0.863
Pyramidal_layer_top20         0.997  0.851
Pyramidal_layer_top50         0.986  0.337
Pyramidal_layer_top100        0.982  0.280
Pyramidal_layer_top200        0.980  0.262
Pyramidal_layer_top500        0.976  0.230
Striatum_top2                 0.917  0.622
Striatum_top5                 0.975  0.787
Striatum_top10                0.977  0.828
Striatum_top20                0.977  0.849
Striatum_top50                0.972  0.844
Stria